In [ ]:
# تابع پارسر سفارشی (Custom Parser)
def parse_final_answer(text: str):
    delimiter = "FINAL_ANSWER:"

    if delimiter in text:
        # متن را از محل جداکننده به دو قسمت تقسیم می‌کنیم
        parts = text.split(delimiter)
        reasoning = parts[0].strip()
        answer = parts[1].strip()

        return {
            "thought_process": reasoning,  # فکر را نگه می‌داریم (شاید برای لاگ)
            "result": answer               # جواب را به کاربر می‌دهیم
        }
    else:
        return {"error": "Could not find final answer", "raw": text}

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://api.tapsage.com/openai/v1",
    api_key="YOUR_API_KEY",
    model="gpt-4o",
    temperature=0
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# پرامپت با دستورالعمل جداکننده
template = """
مسئله زیر را حل کن.
مهم: ابتدا مرحله به مرحله فکر کن (Reasoning).
سپس پاسخ نهایی را دقیقاً بعد از عبارت "FINAL_ANSWER:" بنویس.

مسئله: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ساخت زنجیره
chain = prompt | model | StrOutputParser() | RunnableLambda(parse_final_answer)

# اجرا
output = chain.invoke({"question": "اگر سن من (۲۰ سال) دو برابر برادرم باشد، وقتی من ۴۰ ساله شوم برادرم چند ساله است؟"})

print(f"--- تفکرات مدل ---\n{output['thought_process'][:100]}...") # نمایش خلاصه فکر
print(f"\n--- پاسخ نهایی ---\n{output['result']}")

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# تعریف ساختار داده مطلوب (Schema)
class MathSolution(BaseModel):
    reasoning: str = Field(description="مراحل تفکر و استدلال گام به گام")
    final_answer: float = Field(description="فقط عدد نهایی پاسخ")

# ساخت پارسر
parser = PydanticOutputParser(pydantic_object=MathSolution)


# نکته کلیدی: متد .get_format_instructions() دستورالعمل‌های دقیق JSON را به پرامپت تزریق می‌کند
template = """
مسئله ریاضی زیر را حل کن.
{format_instructions}

مسئله: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# تزریق دستورالعمل‌ها به پرامپت (Partial Formatting)
# این کار باعث می‌شود مدل بفهمد دقیقاً چه فرمت JSONی باید تولید کند
final_prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = final_prompt | model | parser

try:
    result = chain.invoke({"question": "ریشه دوم عدد ۱۴۴ به علاوه ۵ به توان ۲ چند می‌شود؟"})
    
    # خروجی الان یک آبجکت پایتون است! (نه دیکشنری، نه متن)
    print(f"نوع داده خروجی: {type(result)}") 
    print(f"استدلال: {result.reasoning}")
    print(f"عدد نهایی: {result.final_answer}")
    
except Exception as e:
    print(f"خطا در پارس کردن: {e}")

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# تعریف ساختار داده مطلوب (Schema)
class MathSolution(BaseModel):
    reasoning: str = Field(description="مراحل تفکر و استدلال گام به گام")
    final_answer: float = Field(description="فقط عدد نهایی پاسخ")

# ساخت پارسر
parser = PydanticOutputParser(pydantic_object=MathSolution)


# نکته کلیدی: متد .get_format_instructions() دستورالعمل‌های دقیق JSON را به پرامپت تزریق می‌کند
template = """
مسئله ریاضی زیر را حل کن.
{format_instructions}

مسئله: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# تزریق دستورالعمل‌ها به پرامپت (Partial Formatting)
# این کار باعث می‌شود مدل بفهمد دقیقاً چه فرمت JSONی باید تولید کند
final_prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = final_prompt | model | parser

try:
    result = chain.invoke({"question": "ریشه دوم عدد ۱۴۴ به علاوه ۵ به توان ۲ چند می‌شود؟"})
    
    # خروجی الان یک آبجکت پایتون است! (نه دیکشنری، نه متن)
    print(f"نوع داده خروجی: {type(result)}") 
    print(f"استدلال: {result.reasoning}")
    print(f"عدد نهایی: {result.final_answer}")
    
except Exception as e:
    print(f"خطا در پارس کردن: {e}")